# Lesson 2.2 — experiments: BFS and DFS in the warehouse

Run the cells in order. Before each experiment, write your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the experiments before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

The function `run` performs one search on a map and returns the goal node and the list of expanded fields. The function `summary` prints the route length and the number of expansions. In the figures, blue fields were expanded and orange ones form the route.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"


def run(algorithm, warehouse, action_order=("N", "E", "S", "W"), limit=100_000):
    """Run one search on a map; return the goal node and the expanded cells."""
    try:
        with record(limit=limit) as trace:
            node = algorithm(RouteProblem(warehouse, action_order))
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{algorithm.__name__} expanded more than {limit} nodes - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    expanded = [event.data["node"].state for event in trace.of_kind("expand")]
    return node, expanded


def summary(label, node, expanded):
    steps = len(node.solution()) if node else None
    print(f"{label}: route length {steps}, expanded {len(expanded)}")

## Experiment 1: the map with aisles

The robot `R` stands in the top corridor, and the dock `D` is in the bottom-left corner. The robot tries its moves in the order N, E, S, W.

In [ ]:
aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
print(render_text(aisles))

**Your prediction.** Which method finds the shorter route? Which expands more fields? Roughly how many?

*My prediction:* …

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, ["bfs", "dfs"]):
    node, expanded = run(getattr(algorithms, name), aisles)
    summary(name, node, expanded)
    path = node.states() if node else []
    plot_map(aisles, path=path, expanded=expanded, title=name.upper(), ax=ax)
fig.tight_layout()

**Think about it.** Compare the result with your prediction. Why did DFS head east when the dock is to the west? Does DFS "try" to find a short route at all?

## Experiment 2: the order of actions

Same map, but now the robot tries its moves in the order W, S, E, N.

**Your prediction.** How will the BFS route change, and how the DFS route?

*My prediction:* …

In [ ]:
for order in [("N", "E", "S", "W"), ("W", "S", "E", "N")]:
    for name in ["bfs", "dfs"]:
        node, expanded = run(getattr(algorithms, name), aisles, order)
        summary(f"{name}, order {' '.join(order)}", node, expanded)

In [ ]:
node, expanded = run(algorithms.dfs, aisles, ("W", "S", "E", "N"))
path = node.states() if node else []
ax = plot_map(aisles, path=path, expanded=expanded, title="DFS, order W S E N")

**Think about it.** Which method is sensitive to the order of actions, and which is not? Does the good DFS result with this order mean that DFS suits this problem well?

## Experiment 3: open space

The `open_floor` map has almost no obstacles, and the dock is close to the start.

**Your prediction.** Which method expands more fields than in the aisles? How long will the DFS route be?

*My prediction:* …

In [ ]:
open_floor = WarehouseMap.from_file(MAPS / "open_floor.txt")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, ["bfs", "dfs"]):
    node, expanded = run(getattr(algorithms, name), open_floor)
    summary(name, node, expanded)
    path = node.states() if node else []
    plot_map(open_floor, path=path, expanded=expanded, title=name.upper(), ax=ax)
fig.tight_layout()

**Think about it.** Why does BFS expand so many fields here, even though the dock is close? Why is the DFS route even longer than in the aisles? What information would help both algorithms? Lesson 2.4 answers that.

## Summary

Go back to the review questions in the lesson. After your own attempt, you can compare your results and conclusions with the answers in `solutions/02_search/02_bfs_dfs/answers.md`.